# 04A Real COF ML Case Study: CO₂ Adsorption

> 🟢 **Level A · Required**

Integrate 02A–03C: `data → EDA → clean → features → models → CV → final test → interpretation`. The target is published GCMC data, not an experimental measurement.


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline


In [ ]:
import pandas as pd
df=pd.read_csv('https://raw.githubusercontent.com/gokhanonderaksu/COFSpace/main/OnlyCoRECOF%20-%20Feature%20Sets/CoRECOF%20-%20CO2%20-%201%20BAR.csv')
target='CO2-1 bar (mol/kg)'
features=['PLD (Å)','LCD (Å)','Sacc (m2/g-1)','Porosity','%C','%H','%N','%O','%Metalloid','%Halogen','%Ametal']
X=df[features].copy(); X['LCD_PLD_ratio']=X['LCD (Å)']/X['PLD (Å)']; X=X.replace([float('inf'), -float('inf')], float('nan')); y=df[target]
display(df.head())


In [ ]:
from sklearn.model_selection import KFold,cross_validate,train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor,ExtraTreesRegressor,GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error,r2_score
Xdev,Xtest,ydev,ytest=train_test_split(X,y,test_size=0.2,random_state=42)
cv=KFold(5,shuffle=True,random_state=42)
models={'Ridge':make_pipeline(StandardScaler(),Ridge()),'Random Forest':RandomForestRegressor(n_estimators=400,random_state=42,n_jobs=-1),'Extra Trees':ExtraTreesRegressor(n_estimators=400,random_state=42,n_jobs=-1),'Gradient Boosting':GradientBoostingRegressor(random_state=42)}
models={name:make_pipeline(SimpleImputer(strategy='median'),model) for name,model in models.items()}
rows=[]
for name,m in models.items():
    s=cross_validate(m,Xdev,ydev,cv=cv,scoring={'mae':'neg_mean_absolute_error','r2':'r2'}); rows.append([name,(-s['test_mae']).mean(),(-s['test_mae']).std(),s['test_r2'].mean()])
results=pd.DataFrame(rows,columns=['Model','CV_MAE','CV_MAE_std','CV_R2']).sort_values('CV_MAE'); display(results)
best=models[results.iloc[0]['Model']].fit(Xdev,ydev); p=best.predict(Xtest)
print('test MAE =',mean_absolute_error(ytest,p),'test R² =',r2_score(ytest,p))


Use the model-comparison and cross-validation pattern from Chapters 06–07, then report one untouched test result. Repeat with `KCO2` and at different pressures as sensitivity tests.

### Completion criterion
Complete a real supervised-ML workflow independently and document dataset provenance, conditions, descriptors, split, preprocessing and limitations.


## Extension: a second real dataset
Run after the COFSpace baseline. Variables below refer to a separate 30 bar task; do not compare its numerical MAE directly with the 1 bar task.


## 2. Dataset B — CURATED-COFs adsorption properties

`nachatz/cof-data` organizes adsorption tasks derived from CURATED-COFs / Materials Cloud. `properties.csv` contains H₂, O₂, CO₂, CH₄, N₂, Xe, Kr, H₂O and H₂S properties, while `simple_features.csv` contains ASA, density and a pore-size descriptor. The stable COF identifier connects the tables. This is a useful exercise in **joining structure/features and properties by material ID**.


### Audit the join
First check missing/duplicate keys, unmatched rows and target units. An inner join alone hides exclusions. The derivative tables are linked below; cite the original Materials Cloud record as well. Confirm temperature and simulation protocol there before research use.


In [ ]:
prop_url = 'https://raw.githubusercontent.com/nachatz/cof-data/main/properties.csv'
feat_url = 'https://raw.githubusercontent.com/nachatz/cof-data/main/simple_features.csv'
properties = pd.read_csv(prop_url)
simple_features = pd.read_csv(feat_url)
assert simple_features['cof'].notna().all() and properties['name'].notna().all()
assert simple_features['cof'].is_unique and properties['name'].is_unique
coverage = simple_features[['cof']].merge(properties[['name']], left_on='cof', right_on='name', how='outer', indicator=True, validate='one_to_one')
display(coverage['_merge'].value_counts().to_frame('rows'))
display(properties['co2_ads_unit'].value_counts(dropna=False).to_frame('rows'))
assert properties['co2_ads_unit'].dropna().nunique() == 1, 'Resolve mixed target units before training'
dataset_b = simple_features.merge(properties, left_on='cof', right_on='name', how='inner', validate='one_to_one')
print('features:', simple_features.shape, 'properties:', properties.shape, 'merged:', dataset_b.shape)
display(dataset_b[['cof','ASA_m^2/g','Density','LS','co2_30bar','co2_ads_unit','co2_henry','co2_henry_unit']].head())


In [ ]:
df = dataset_b[['ASA_m^2/g','Density','LS','co2_30bar']].dropna()
X, y = df[['ASA_m^2/g','Density','LS']], df['co2_30bar']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
m2 = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1).fit(X_train, y_train)
p2 = m2.predict(X_test)
print('n =', len(df), 'MAE =', mean_absolute_error(y_test, p2), 'R2 =', r2_score(y_test, p2))


## 4. What does each dataset teach?

| Dataset | Type | Best teaching use |
|---|---|---|
| COFSpace / CoRE COF | ~10³, simulated labels | supervised regression, feature importance, gas/pressure-dependent targets |
| CURATED-COFs adsorption | experimental COF structures + computed properties | CIF/ID/property joins, units and conditions, small-data baselines |
| ReDD-COFFEE HTS | ~10⁵ hypothetical COFs | fixed splits, feature reduction, surrogate screening, SHAP and scale |

Do not concatenate these datasets blindly. Their structure sources, simulation protocols, descriptor definitions and target conditions differ. Treat cross-dataset use as transfer learning, external validation or domain-shift analysis only after checking compatibility.


## 5. Exercises

1. Compare CO₂ models at 0.1, 1, 5 and 10 bar in COFSpace.
2. Predict both `co2_30bar` and `co2_henry` in Dataset B and compare controlling features.
3. Merge composition descriptors calculated from CIFs in 04B with Dataset B through COF IDs.
4. For ReDD-COFFEE, use the authors' fixed train/test lists rather than generating a new random split.
5. Write a dataset card for every experiment: source, structure type, target, T/P, units, simulation method, features, split and citation/license.


## Sources and citation

- COFSpace: G. Onder Aksu et al., *The COF Space: Materials Features, Gas Adsorption, and Separation Performances Assessed by Machine Learning*. Data/scripts: https://github.com/gokhanonderaksu/COFSpace
- CURATED-COFs / Materials Cloud: D. Ongari et al., *Building a consistent and reproducible database for adsorption evaluation in Covalent-Organic Frameworks*. Data DOI: 10.24435/materialscloud:z6-jn.
- ReDD-COFFEE CO₂-capture HTS: https://github.com/jsdvos/SupportingInformation_CO2captureHTS_2024

For research use, cite the original papers and dataset records rather than only this tutorial.


## Sources and further reading
[Dataset contracts / 数据使用约定](../../docs/data_resources.md) · [COFSpace](https://github.com/gokhanonderaksu/COFSpace) · [CURATED-COFs](https://github.com/danieleongari/CURATED-COFs)
